# Advanced RAG Pipeline — Overview

This notebook demonstrates an advanced Retrieval-Augmented Generation (RAG) pipeline using LangChain and LangSmith. It loads tools (Wikipedia, ArXiv, and a document retriever), configures an LLM, and creates an agent that can call tools to answer user questions. 

Key points:
- Ensure LangSmith endpoint is correct to pull shared prompts from the Global Hub.
- If a Hub prompt is unavailable (404), the notebook falls back to a local prompt template so the pipeline remains functional.

Steps covered:
1. Load helper tools and wrappers (Wikipedia, ArXiv, web loader + FAISS retriever).
2. Configure LLM and LangSmith client.
3. Pull hub prompt (with robust fallback if not found).
4. Create an agent and run example queries.

Documentation & tips:
- If you see a 404 when pulling a hub prompt, set `os.environ['LANGSMITH_ENDPOINT']` to `https://api.smith.langchain.com` to ensure you target the global hub instead of a regional one.
- The notebook includes inline comments explaining each step.

In [47]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [48]:
api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=200)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper)
wiki.name

'wikipedia'

In [49]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader=WebBaseLoader("https://docs.smith.langchain.com/")
docs=loader.load()
documents=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
vectordb=FAISS.from_documents(documents, OpenAIEmbeddings())
retriever=vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000021B082A8F10>, search_kwargs={})

In [50]:
from langchain_core.tools.retriever import create_retriever_tool
retriever_tool=create_retriever_tool(retriever,"langsmith_search","Search for information about LangSmith. For any questions about LangSmith, you must use this tool!")
retriever_tool.name


'langsmith_search'

In [51]:
## Arxiv Tool
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=200)
arxiv=ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv.name

'arxiv'

In [52]:
tools=[wiki,arxiv,retriever_tool]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\Pritam\\Desktop\\LangChain\\venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=200)),
 StructuredTool(name='langsmith_search', description='Search for information about LangSmith. For any questions about LangSmith, you must use this tool!', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x0000021B082F4FE0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x0000021B082F5580>)]

In [53]:
from dotenv import load_dotenv

# Load env vars
load_dotenv()

from langchain_ollama import OllamaLLM
from langchain_openai import ChatOpenAI

# LLM
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

In [54]:
import os
from langsmith import Client
from langchain_core.prompts import ChatPromptTemplate

# Ensure we point to the global LangSmith hub (overrides .env) to avoid region-specific 404s.
# Some clients read different env vars (LANGSMITH_ENDPOINT or LANGCHAIN_ENDPOINT),
# so set both to be safe.
os.environ["LANGSMITH_ENDPOINT"] = 'https://api.smith.langchain.com'
os.environ["LANGCHAIN_ENDPOINT"] = 'https://api.smith.langchain.com'

# Print the effective endpoints for debugging/verification
print("LANGSMITH_ENDPOINT:", os.environ.get("LANGSMITH_ENDPOINT"))
print("LANGCHAIN_ENDPOINT:", os.environ.get("LANGCHAIN_ENDPOINT"))

client = Client()

try:
    # Try to pull the shared prompt from the Global Hub
    prompt = client.pull_prompt("hwchase17/openai-functions-agent")
    print(f"Prompt loaded: {len(prompt.messages)} messages")
except Exception as e:
    # Fallback strategy:
    # 1) Try a direct HTTP GET to the Global Hub commit manifest and reconstruct the prompt.
    # 2) If that fails (403, parsing error, network issues), fall back to a robust local prompt template.
    print(f"Warning: failed to pull hub prompt ({e}). Attempting direct HTTP fetch of manifest from global hub...")
    try:
        import requests
        url = "https://api.smith.langchain.com/commits/hwchase17/openai-functions-agent/latest"
        r = requests.get(url, timeout=10)
        if r.ok:
            data = r.json()
            manifest = data.get("manifest", {})
            messages = manifest.get("kwargs", {}).get("messages", [])

            from langchain_core.prompts import (
                ChatPromptTemplate,
                SystemMessagePromptTemplate,
                HumanMessagePromptTemplate,
                MessagesPlaceholder,
            )

            lm_messages = []
            for m in messages:
                id_path = m.get("id", [])
                kind = id_path[-1] if id_path else None
                if kind == "SystemMessagePromptTemplate":
                    template = m["kwargs"]["prompt"]["kwargs"]["template"]
                    lm_messages.append(SystemMessagePromptTemplate.from_template(template))
                elif kind == "HumanMessagePromptTemplate":
                    template = m["kwargs"]["prompt"]["kwargs"]["template"]
                    lm_messages.append(HumanMessagePromptTemplate.from_template(template))
                elif kind == "MessagesPlaceholder":
                    opt = m["kwargs"].get("optional", False)
                    var = m["kwargs"].get("variable_name", "chat_history")
                    lm_messages.append(MessagesPlaceholder(variable_name=var, optional=opt))
                else:
                    # Unknown or unsupported message type — skip
                    pass

            # Build ChatPromptTemplate from reconstructed messages
            prompt = ChatPromptTemplate.from_messages(lm_messages)
            print("Prompt reconstructed from hub manifest via direct HTTP fetch.")
        else:
            print("Direct HTTP fetch failed with status", r.status_code, ". Falling back to local prompt.")
            raise RuntimeError("manifest fetch failed")
    except Exception as ex2:
        print("Could not reconstruct prompt from manifest:", ex2)
        # Final robust local fallback
        fallback_text = """You are an assistant that can call external tools to answer user queries.
- When you call a tool, examine the tool output and incorporate it into your answer.
- If the tool returns documents or source links, cite them in the response.
- Be concise and helpful; clarify when info is missing.

User question:
{input}
"""
        prompt = ChatPromptTemplate.from_template(fallback_text)
        print("Using local fallback prompt.")

LANGSMITH_ENDPOINT: https://api.smith.langchain.com
LANGCHAIN_ENDPOINT: https://api.smith.langchain.com
Prompt reconstructed from hub manifest via direct HTTP fetch.


In [55]:
# Inspect Client internals to understand which base URL it uses and try a direct GET to the commits endpoint
print("client repr:", client)
# Show commonly used attributes
for attr in ["base_url","base_url","_base_url","_client","http","_http","client","api_base_url"]:
    if hasattr(client, attr):
        print(attr,":", getattr(client, attr))

# If the client exposes an http session object, try to fetch the commit endpoint directly.
try:
    import requests
    url = "https://api.smith.langchain.com/commits/hwchase17/openai-functions-agent/latest"
    r = requests.get(url, timeout=10)
    print("Direct GET status:", r.status_code)
    print(r.text[:1000])
except Exception as e:
    print("Direct GET failed:", e)

client repr: Client (API URL: https://eu.api.smith.langchain.com)
Direct GET status: 200
{"commit_hash":"a1655024b06afbd95d17449f21316291e0726f13dcfaf990cc0d18087ad689a5","manifest":{"id":["langchain","prompts","chat","ChatPromptTemplate"],"lc":1,"type":"constructor","kwargs":{"messages":[{"id":["langchain","prompts","chat","SystemMessagePromptTemplate"],"lc":1,"type":"constructor","kwargs":{"prompt":{"id":["langchain","prompts","prompt","PromptTemplate"],"lc":1,"type":"constructor","kwargs":{"template":"You are a helpful assistant","input_variables":[],"template_format":"f-string","partial_variables":{}}}}},{"id":["langchain","prompts","chat","MessagesPlaceholder"],"lc":1,"type":"constructor","kwargs":{"optional":true,"variable_name":"chat_history"}},{"id":["langchain","prompts","chat","HumanMessagePromptTemplate"],"lc":1,"type":"constructor","kwargs":{"prompt":{"id":["langchain","prompts","prompt","PromptTemplate"],"lc":1,"type":"constructor","kwargs":{"template":"{input}","input_var

In [56]:
import inspect
print(inspect.signature(Client))
print(Client.__doc__)
# Try to instantiate Client with a base_url kwarg if supported
try:
    client_explicit = Client(base_url="https://api.smith.langchain.com")
    print("Explicit client repr:", client_explicit)
    # try pulling the prompt with explicit client
    p = client_explicit.pull_prompt("hwchase17/openai-functions-agent")
    print("Explicit pull success, messages:", len(p.messages))
except Exception as exc:
    print("Explicit client attempt failed:", exc)

(api_url: 'Optional[str]' = None, *, api_key: 'Optional[str]' = None, retry_config: 'Optional[Retry]' = None, timeout_ms: 'Optional[Union[int, tuple[int, int]]]' = None, web_url: 'Optional[str]' = None, session: 'Optional[requests.Session]' = None, auto_batch_tracing: 'bool' = True, anonymizer: 'Optional[Callable[[dict], dict]]' = None, hide_inputs: 'Optional[Union[Callable[[dict], dict], bool]]' = None, hide_outputs: 'Optional[Union[Callable[[dict], dict], bool]]' = None, hide_metadata: 'Optional[Union[Callable[[dict], dict], bool]]' = None, omit_traced_runtime_info: 'bool' = False, process_buffered_run_ops: 'Optional[Callable[[Sequence[dict]], Sequence[dict]]]' = None, run_ops_buffer_size: 'Optional[int]' = None, run_ops_buffer_timeout_ms: 'Optional[float]' = None, info: 'Optional[Union[dict, ls_schemas.LangSmithInfo]]' = None, api_urls: 'Optional[dict[str, str]]' = None, otel_tracer_provider: 'Optional[TracerProvider]' = None, otel_enabled: 'Optional[bool]' = None, tracing_sampling_

In [57]:
# Try creating a Client instance with explicit api_url to force use of global hub
try:
    client_global = Client(api_url="https://api.smith.langchain.com")
    print("Explicit client repr:", client_global)
    p = client_global.pull_prompt("hwchase17/openai-functions-agent")
    print("Explicit pull success, messages:", len(p.messages))
except Exception as exc:
    print("Explicit client attempt failed:", exc)

Explicit client repr: Client (API URL: https://api.smith.langchain.com)
Explicit client attempt failed: Failed to GET /commits/hwchase17/openai-functions-agent/latest in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/commits/hwchase17/openai-functions-agent/latest', '{"error":"Forbidden"}\n')


In [58]:
### Agents
from langchain_classic.agents import create_openai_tools_agent
agent=create_openai_tools_agent(llm,tools,prompt)

In [59]:
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)

In [60]:
from langchain_classic.agents import AgentExecutor
agent_executor=AgentExecutor(agent=agent, tools=tools,verbose=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag

In [61]:
agent_executor.invoke({"input":"Tell me about Langsmith"})



> Entering new AgentExecutor chain...

Invoking: `langsmith_search` with `{'query': 'Langsmith'}`


LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringDeploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansAccount administrationOverviewSet up hierarchyWorkload isolationManage organizations using the APIManage billingGranular usageSet up resource tagsUser managementAdditional resourcesPolly (Beta)Data managementAccess control & AuthenticationScalability & resilienceFAQsRegions FAQPricing FAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith provides tools for developing, debugging, and deploying LLM applications.
It helps you trace requests, evaluate outputs, test prompts, and manage deployments in one place.
LangSmith is framework agnostic, so you can use it with or without L

{'input': 'Tell me about Langsmith',
 'output': "LangSmith provides tools for developing, debugging, and deploying LLM (Large Language Model) applications. It helps trace requests, evaluate outputs, test prompts, and manage deployments in one place. LangSmith is framework agnostic, allowing you to use it with or without LangChain’s open-source libraries like langchain and langgraph.\n\nYou can prototype locally and then move to production with integrated monitoring and evaluation to build more reliable AI systems. The LangGraph Platform has been rebranded as LangSmith Deployment. To get started with LangSmith, you can create an account, generate an API key, choose your integration, and set up the platform according to your infrastructure and compliance needs.\n\nLangSmith offers features like Studio for visual interface application design, prompt testing, and Agent Builder for designing and deploying AI agents visually without writing code. It ensures data security and privacy complian

In [62]:
agent_executor.invoke({"input":"What's the paper 1605.08386 about?"})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1605.08386'}`




c:\Users\Pritam\Desktop\LangChain\venv\Lib\site-packages\langchain_community\utilities\arxiv.py:102: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  ).results()


Published: 2016-05-26
Title: Heat-bath random walks with Markov bases
Authors: Caprice Stanley, Tobias Windisch
Summary: Graphs on lattice points are studied whose edges come from a finite set of alloThe paper with identifier 1605.08386 is titled "Heat-bath random walks with Markov bases" by Caprice Stanley and Tobias Windisch. It discusses the study of graphs on lattice points where the edges come from a finite set of allo.

> Finished chain.


{'input': "What's the paper 1605.08386 about?",
 'output': 'The paper with identifier 1605.08386 is titled "Heat-bath random walks with Markov bases" by Caprice Stanley and Tobias Windisch. It discusses the study of graphs on lattice points where the edges come from a finite set of allo.'}